In [0]:
#wczytanie tabeli
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("dbfs:/FileStore/tables/trip/201508_trip_data.csv")
df.createOrReplaceTempView("dfTable")
display(df)


Trip ID,Duration,Start Date,Start Station,Start Terminal,End Date,End Station,End Terminal,Bike #,Subscriber Type,Zip Code
913460,765,8/31/2015 23:26,Harry Bridges Plaza (Ferry Building),50,8/31/2015 23:39,San Francisco Caltrain (Townsend at 4th),70,288,Subscriber,2139
913459,1036,8/31/2015 23:11,San Antonio Shopping Center,31,8/31/2015 23:28,Mountain View City Hall,27,35,Subscriber,95032
913455,307,8/31/2015 23:13,Post at Kearny,47,8/31/2015 23:18,2nd at South Park,64,468,Subscriber,94107
913454,409,8/31/2015 23:10,San Jose City Hall,10,8/31/2015 23:17,San Salvador at 1st,8,68,Subscriber,95113
913453,789,8/31/2015 23:09,Embarcadero at Folsom,51,8/31/2015 23:22,Embarcadero at Sansome,60,487,Customer,9069
913452,293,8/31/2015 23:07,Yerba Buena Center of the Arts (3rd @ Howard),68,8/31/2015 23:12,San Francisco Caltrain (Townsend at 4th),70,538,Subscriber,94118
913451,896,8/31/2015 23:07,Embarcadero at Folsom,51,8/31/2015 23:22,Embarcadero at Sansome,60,363,Customer,92562
913450,255,8/31/2015 22:16,Embarcadero at Sansome,60,8/31/2015 22:20,Steuart at Market,74,470,Subscriber,94111
913449,126,8/31/2015 22:12,Beale at Market,56,8/31/2015 22:15,Temporary Transbay Terminal (Howard at Beale),55,439,Subscriber,94130
913448,932,8/31/2015 21:57,Post at Kearny,47,8/31/2015 22:12,South Van Ness at Market,66,472,Subscriber,94702


In [0]:
display(df.summary())

summary,Trip ID,Duration,Start Date,Start Station,Start Terminal,End Date,End Station,End Terminal,Bike #,Subscriber Type,Zip Code
count,354152,354152,354152,354152,354152,354152,354152,354152,354152,354152,353874
mean,676962.27441607,1046.0326611172604,null,null,58.4460175291965,null,null,58.42181605638257,422.9579107275972,null,2613469.697174148
stddev,138874.1535127196,30016.936156929794,null,null,16.73886047969254,null,null,16.87679234429513,159.84155003059428,null,1.4617651129228117E9
min,432947,60,1/1/2015 0:25,2nd at Folsom,2,1/1/2015 0:30,2nd at Folsom,2,9,Customer,0
25%,556964,342,null,null,50,null,null,50,327,null,94103.0
50%,679429,511,null,null,62,null,null,63,437,null,94111.0
75%,797997,739,null,null,70,null,null,70,546,null,94566.0
max,913460,17270400,9/9/2014 9:59,Yerba Buena Center of the Arts (3rd @ Howard),84,9/9/2014 9:59,Yerba Buena Center of the Arts (3rd @ Howard),84,878,Subscriber,nil


In [0]:
#REGEX
#regexp_replace
#zamiana konkretnego wzorca 
from pyspark.sql.functions import regexp_replace, col,regexp_extract,when

place = "Embarcadero at Sansome|Embarcadero at Bryant|Embarcadero at Vallejo"
#zamiana stacji na ta sama naze(polozone blisko siebie)
df = df.withColumn("End Station_replace",regexp_replace(col("End Station"), place, "Embarcadero"))

df.show()

#regexp_extract 
#do wyciągnięcia konkretnej wartości z kolumny
df = df.withColumn("Second_Part_of_Station", when(col("End Station").rlike(place),regexp_extract(col("End Station"), r"at\s+(.+)", 1)).otherwise(None))

df.select("End Station", "Second_Part_of_Station").show(truncate=False)


+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+
| 913460|     765|8/31/2015 23:26|Harry Bridges Pla...|            50|8/31/2015 23:39|San Francisco Cal...|          70|   288|     Subscriber|    2139|San Francisco Cal...|
| 913459|    1036|8/31/2015 23:11|San Antonio Shopp...|            31|8/31/2015 23:28|Mountain View Cit...|          27|    35|     Subscriber|   95032|Mountain View Cit...|
| 913455|     307|8/31/2015 23:13|      Post at Kearny|            47|8/31/2015 23:18|   2nd at South Park|          64|   468|   

In [0]:
from pyspark.sql.functions import col
#gdzie zip code jest null
df.select("*").where(col("Zip Code").isNull()).show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+
| 912455|    2092|8/31/2015 12:40|SJSU 4th at San C...|            12|8/31/2015 13:14|SJSU 4th at San C...|          12|   186|       Customer|    NULL|SJSU 4th at San C...|                  NULL|
| 910752|     586| 8/29/2015 9:01|Harry Bridges Pla...|            50| 8/29/2015 9:10|Powell at Post (U...|          71|   260|       Customer|    NULL|Powell at Post (U...|                  NULL|
| 910713|    43

In [0]:
#FILL
from pyspark.sql.functions import filter
df = df.fillna({"Zip code": "No code"})
df.filter(col("Zip code") == "No code").show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+
| 912455|    2092|8/31/2015 12:40|SJSU 4th at San C...|            12|8/31/2015 13:14|SJSU 4th at San C...|          12|   186|       Customer| No code|SJSU 4th at San C...|                  NULL|
| 910752|     586| 8/29/2015 9:01|Harry Bridges Pla...|            50| 8/29/2015 9:10|Powell at Post (U...|          71|   260|       Customer| No code|Powell at Post (U...|                  NULL|
| 910713|    43

In [0]:
#DROP
#do usuniecia calej kolumny lub wierszy z null
df_cleaned = df.dropna(subset=["Second_Part_of_Station"])
df_cleaned_v2 = df.drop("End Terminal")
df_cleaned.show()
df_cleaned_v2.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+-------------------+----------------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code|End Station_replace|Second_Part_of_Station|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+-------------------+----------------------+
| 913453|     789|8/31/2015 23:09|Embarcadero at Fo...|            51|8/31/2015 23:22|Embarcadero at Sa...|          60|   487|       Customer|    9069|        Embarcadero|               Sansome|
| 913451|     896|8/31/2015 23:07|Embarcadero at Fo...|            51|8/31/2015 23:22|Embarcadero at Sa...|          60|   363|       Customer|   92562|        Embarcadero|               Sansome|
| 913415|     274|8/

In [0]:
#EXPLODE
from pyspark.sql.functions import array,explode,split

#do rozbicia tablicy na osobne wiersze
#najpierw musimy konwertowac jakas kolumne na tablice poniewaz zadnej nie mamy
df_split = df.withColumn("Start_Split", split(col("Start Date"), " "))
df_split = df_split.withColumn("Start_Array", array(col("Start_Split").getItem(0), col("Start_Split").getItem(1)))
df_exploded = df_split.withColumn("Explode", explode(col("Start_Array")))
df_exploded.select("Start Date", "Start_Array", "Explode").show(truncate=False)

+---------------+------------------+---------+
|Start Date     |Start_Array       |Explode  |
+---------------+------------------+---------+
|8/31/2015 23:26|[8/31/2015, 23:26]|8/31/2015|
|8/31/2015 23:26|[8/31/2015, 23:26]|23:26    |
|8/31/2015 23:11|[8/31/2015, 23:11]|8/31/2015|
|8/31/2015 23:11|[8/31/2015, 23:11]|23:11    |
|8/31/2015 23:13|[8/31/2015, 23:13]|8/31/2015|
|8/31/2015 23:13|[8/31/2015, 23:13]|23:13    |
|8/31/2015 23:10|[8/31/2015, 23:10]|8/31/2015|
|8/31/2015 23:10|[8/31/2015, 23:10]|23:10    |
|8/31/2015 23:09|[8/31/2015, 23:09]|8/31/2015|
|8/31/2015 23:09|[8/31/2015, 23:09]|23:09    |
|8/31/2015 23:07|[8/31/2015, 23:07]|8/31/2015|
|8/31/2015 23:07|[8/31/2015, 23:07]|23:07    |
|8/31/2015 23:07|[8/31/2015, 23:07]|8/31/2015|
|8/31/2015 23:07|[8/31/2015, 23:07]|23:07    |
|8/31/2015 22:16|[8/31/2015, 22:16]|8/31/2015|
|8/31/2015 22:16|[8/31/2015, 22:16]|22:16    |
|8/31/2015 22:12|[8/31/2015, 22:12]|8/31/2015|
|8/31/2015 22:12|[8/31/2015, 22:12]|22:12    |
|8/31/2015 21

In [0]:
#IFNULL
from pyspark.sql.functions import ifnull,lit
df = df.withColumn("End Station Final", ifnull(col("Second_Part_of_Station"), lit("Don't have")))
df.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|End Station Final|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+
| 913460|     765|8/31/2015 23:26|Harry Bridges Pla...|            50|8/31/2015 23:39|San Francisco Cal...|          70|   288|     Subscriber|    2139|San Francisco Cal...|                  NULL|       Don't have|
| 913459|    1036|8/31/2015 23:11|San Antonio Shopp...|            31|8/31/2015 23:28|Mountain View Cit...|          27|    35|     Subscrib

In [0]:
#NULLIF
from pyspark.sql.functions import nullif
df= df.withColumn("with_nulls", nullif(col("End Station Final"), lit("Don't have")))
df.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|End Station Final|with_nulls|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+
| 913460|     765|8/31/2015 23:26|Harry Bridges Pla...|            50|8/31/2015 23:39|San Francisco Cal...|          70|   288|     Subscriber|    2139|San Francisco Cal...|                  NULL|       Don't have|      NULL|
| 913459|    1036|8/31/2015 23:11|San Antonio Shopp...|            31|8/31/2015 23:28|Mountain V

In [0]:
#REPLACE
#zamiana wartości inna wartością 
df = df.na.replace({462: 195}, subset=["Bike #"]).filter(col("Zip Code") == '94107')
df.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|End Station Final|with_nulls|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+
| 913455|     307|8/31/2015 23:13|      Post at Kearny|            47|8/31/2015 23:18|   2nd at South Park|          64|   468|     Subscriber|   94107|   2nd at South Park|                  NULL|       Don't have|      NULL|
| 913442|     633|8/31/2015 21:44|      Market at 10th|            67|8/31/2015 21:54|San Franci

In [0]:
#ARRAY CONTAINS
from pyspark.sql.functions import array_contains

#sprawdza czy dana tablica zawiera okreslona wartosc 
df_split = df_split.withColumn("Contains_Date", array_contains(col("Start_Array"), "8/31/2015"))
df_split.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+------------------+------------------+-------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|       Start_Split|       Start_Array|Contains_Date|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+------------------+------------------+-------------+
| 913460|     765|8/31/2015 23:26|Harry Bridges Pla...|            50|8/31/2015 23:39|San Francisco Cal...|          70|   288|     Subscriber|    2139|San Francisco Cal...|                  NULL|[8/31/2015, 23:26]|[8/31/2015, 23:26]|         true|
| 91

In [0]:
#funkcje agregujące 
from pyspark.sql.functions import avg
df.agg(avg("Duration").alias("Average_Duration")).show()

+----------------+
|Average_Duration|
+----------------+
|613.611020548239|
+----------------+



In [0]:
from pyspark.sql.functions import count
df.groupBy("Start Date").agg(count("Trip ID").alias("Trips_per_Day")).show()

+---------------+-------------+
|     Start Date|Trips_per_Day|
+---------------+-------------+
|8/27/2015 22:56|            1|
|8/27/2015 14:00|            2|
| 8/26/2015 8:57|            1|
|8/25/2015 14:47|            1|
|8/24/2015 18:19|            1|
|8/24/2015 10:05|            1|
|8/23/2015 14:23|            1|
| 8/22/2015 8:48|            1|
|8/21/2015 11:54|            1|
| 8/18/2015 8:39|            2|
|8/17/2015 18:10|            1|
|8/14/2015 19:44|            1|
|8/14/2015 16:57|            1|
| 8/13/2015 7:50|            1|
|8/12/2015 18:05|            1|
|8/10/2015 15:18|            1|
|  8/6/2015 8:00|            1|
| 8/3/2015 12:03|            1|
|7/30/2015 17:52|            1|
| 7/30/2015 8:46|            1|
+---------------+-------------+
only showing top 20 rows



In [0]:
df.groupBy("Start Station").agg(count("Trip ID").alias("Trips_from_Station")).orderBy("Trips_from_Station", ascending=False).show(1)

+--------------------+------------------+
|       Start Station|Trips_from_Station|
+--------------------+------------------+
|San Francisco Cal...|              5192|
+--------------------+------------------+
only showing top 1 row



In [0]:
#ZADANIE 2
#UTF-do pisania funkcji ktore pozwola na wlasne transformacje 
#funkcja do liczenia miejsca
from pyspark.sql.types import IntegerType, StringType

#funkcja do obliczenia nowego id
def id_trip(tripid):
    if tripid > 913000:
        return tripid * 2;
    else:
        return tripid;

#rejestracja udf
id_udf = udf(id_trip, IntegerType())
#użycie udf w dataframe
df = df.withColumn('New Id', id_udf(col('Trip ID')))
df.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+-------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|End Station Final|with_nulls| New Id|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+-------+
| 913455|     307|8/31/2015 23:13|      Post at Kearny|            47|8/31/2015 23:18|   2nd at South Park|          64|   468|     Subscriber|   94107|   2nd at South Park|                  NULL|       Don't have|      NULL|1826910|
| 913442|     633|8/31/2015 21:44|      Market at 10th|         

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
#nie trzeba dodatkowo rejestrowac tej funkcji udf
@pandas_udf(StringType())

#dodanie prefiksu place:
def add(place: pd.Series) -> pd.Series:
    return "Place: " + place

df = df.withColumn('place', add(col('Start Station')))
df.show()

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+-------+--------------------+
|Trip ID|Duration|     Start Date|       Start Station|Start Terminal|       End Date|         End Station|End Terminal|Bike #|Subscriber Type|Zip Code| End Station_replace|Second_Part_of_Station|End Station Final|with_nulls| New Id|               place|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+--------------------+----------------------+-----------------+----------+-------+--------------------+
| 913455|     307|8/31/2015 23:13|      Post at Kearny|            47|8/31/2015 23:18|   2nd at South Park|          64|   468|     Subscriber|   94107|   2nd at South Park|                  NULL|       Don't have|      NULL|1826910|Pl